# Tech Challenge Fase 2  
## Notebook 08 — IA e Modelagem

### Responsabilidade do notebook

Este notebook demonstra como a camada Gold pode ser utilizada em aplicações de Inteligência Artificial.

A etapa contempla:

- preparação da base de Machine Learning;
- análise exploratória;
- seleção de features;
- classificação do risco de não atingir a meta;
- avaliação do modelo;
- importância das variáveis;
- clusterização de municípios;
- criação de perfis de vulnerabilidade educacional;
- geração de matriz de risco;
- persistência dos resultados;
- exportação para Power BI.

---

### Entradas

```text
gold/base_modelo_ia/BASE_MODELO_IA_VALIDOS.csv
gold/indicadores/GOLD_INDICADORES.csv
```

### Saídas

```text
gold/ia/model_results
gold/ia/feature_importance
gold/ia/clusters
gold/ia/risk_matrix
gold/exports_powerbi/ia_dashboard
logs/ia/model_metrics
```

## 1. Contexto de IA no projeto

A camada Gold permite utilizar dados tratados e integrados para responder perguntas como:

- quais municípios possuem maior risco de não atingir a meta;
- quais variáveis estão mais relacionadas ao risco;
- quais municípios possuem perfis semelhantes;
- quais grupos apresentam maior vulnerabilidade educacional;
- quais regiões devem ser priorizadas por políticas públicas.

Neste projeto serão utilizadas duas abordagens:

### Aprendizado supervisionado

```text
Random Forest Classifier
```

Objetivo:

```text
prever risco_nao_atingir_meta
```

### Aprendizado não supervisionado

```text
K-Means
```

Objetivo:

```text
agrupar municípios com perfis semelhantes
```

## 2. Importação das bibliotecas

In [0]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    silhouette_score
)

## 3. Leitura das configurações oficiais

In [0]:
CONFIG_FILE_PATH = "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/config/config.json"

config = json.loads(
    Path(CONFIG_FILE_PATH).read_text(
        encoding="utf-8"
    )
)

GOLD_PATH = Path(
    config["paths"]["gold_path"]
)

LOG_PATH = Path(
    config["paths"]["log_path"]
)

EXECUTION_DATE = (
    config["project"]["execution_date"]
)

IA_ROOT_PATH = (
    GOLD_PATH / "ia"
)

IA_MODEL_RESULTS_PATH = (
    IA_ROOT_PATH / "model_results"
)

IA_FEATURE_IMPORTANCE_PATH = (
    IA_ROOT_PATH / "feature_importance"
)

IA_CLUSTERS_PATH = (
    IA_ROOT_PATH / "clusters"
)

IA_RISK_MATRIX_PATH = (
    IA_ROOT_PATH / "risk_matrix"
)

IA_POWERBI_PATH = (
    GOLD_PATH
    / "exports_powerbi"
    / "ia_dashboard"
)

IA_LOG_PATH = (
    LOG_PATH
    / "ia"
    / "model_metrics"
)

for path in [
    IA_MODEL_RESULTS_PATH,
    IA_FEATURE_IMPORTANCE_PATH,
    IA_CLUSTERS_PATH,
    IA_RISK_MATRIX_PATH,
    IA_POWERBI_PATH,
    IA_LOG_PATH
]:
    path.mkdir(
        parents=True,
        exist_ok=True
    )

print("IA_ROOT_PATH:", IA_ROOT_PATH)
print("EXECUTION_DATE:", EXECUTION_DATE)

## 4. Leitura da base de Machine Learning

A base válida contém apenas registros completos para modelagem.

In [0]:
ML_FILE_PATH = (
    GOLD_PATH
    / "base_modelo_ia"
    / "BASE_MODELO_IA_VALIDOS.csv"
)

df_ml = pd.read_csv(
    ML_FILE_PATH,
    sep=";",
    decimal=",",
    encoding="utf-8",
    low_memory=False
)

print(
    "Dimensão da base:",
    df_ml.shape
)

display(
    df_ml.head()
)

## 5. Validação da base

Antes da modelagem, validamos:

- existência do target;
- ausência de duplicidades;
- presença de features numéricas;
- quantidade mínima de registros;
- presença das duas classes do target.

In [0]:
target_column = (
    "risco_nao_atingir_meta"
)

required_columns = [
    "ano",
    "id_municipio",
    "sigla_uf",
    target_column
]

missing_required = [
    column
    for column in required_columns
    if column not in df_ml.columns
]

if missing_required:
    raise KeyError(
        "Colunas obrigatórias ausentes: "
        + ", ".join(
            missing_required
        )
    )

if df_ml.duplicated(
    subset=[
        "ano",
        "id_municipio"
    ]
).any():
    raise ValueError(
        "A base de IA possui duplicidade "
        "por ano e município."
    )

df_ml[target_column] = (
    pd.to_numeric(
        df_ml[target_column],
        errors="coerce"
    )
    .astype("Int64")
)

df_ml = (
    df_ml[
        df_ml[target_column]
        .isin([0, 1])
    ]
    .copy()
)

if len(df_ml) < 50:
    raise ValueError(
        "A base possui poucos registros "
        "para uma modelagem demonstrativa."
    )

if df_ml[target_column].nunique() < 2:
    raise ValueError(
        "O target possui apenas uma classe."
    )

print(
    "Base validada com sucesso."
)

print(
    "Distribuição do target:"
)

display(
    df_ml[target_column]
    .value_counts(dropna=False)
    .rename_axis("classe")
    .reset_index(name="quantidade")
)

## 6. Seleção das features candidatas

As features são escolhidas entre os indicadores disponíveis na Gold.

In [0]:
feature_candidates = [
    "PC_ALUNO_ALFABETIZADO",
    "VL_MEDIA_LP",
    "META_FINAL_2030",
    "gap_meta_2030",
    "qtd_alunos",
    "qtd_escolas",
    "qtd_presentes_lp",
    "qtd_alfabetizados",
    "taxa_participacao_lp",
    "taxa_alfabetizacao_alunos",
    "proficiencia_media_lp"
]

features = [
    column
    for column in feature_candidates
    if column in df_ml.columns
]

if len(features) < 3:
    raise ValueError(
        "A base possui poucas features "
        "numéricas disponíveis."
    )

for column in features:
    df_ml[column] = pd.to_numeric(
        df_ml[column],
        errors="coerce"
    )

print("Features utilizadas:")
print(features)

## 7. Análise exploratória

Nesta etapa observamos:

- estatísticas descritivas;
- percentual de nulos;
- distribuição do target;
- correlação entre variáveis.

In [0]:
df_descritiva = (
    df_ml[features]
    .describe()
    .T
    .reset_index()
    .rename(
        columns={
            "index": "feature"
        }
    )
)

display(df_descritiva)

df_nulos = (
    df_ml[features]
    .isna()
    .mean()
    .mul(100)
    .reset_index()
)

df_nulos.columns = [
    "feature",
    "percentual_nulos"
]

display(
    df_nulos
    .sort_values(
        "percentual_nulos",
        ascending=False
    )
)

## 8. Preparação dos dados para classificação

In [0]:
X = df_ml[features].copy()
y = df_ml[target_column].astype(int)

X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y
    )
)

print(
    "Treino:",
    X_train.shape
)

print(
    "Teste:",
    X_test.shape
)

## 9. Pipeline de classificação

O pipeline inclui:

- imputação de valores ausentes;
- Random Forest;
- balanceamento por peso de classe;
- semente fixa para reprodutibilidade.

In [0]:
classification_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_split=4,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

classification_pipeline.fit(
    X_train,
    y_train
)

print(
    "Modelo treinado com sucesso."
)

## 10. Avaliação do modelo

In [0]:
y_pred = (
    classification_pipeline
    .predict(X_test)
)

y_prob = (
    classification_pipeline
    .predict_proba(X_test)[:, 1]
)

metrics = {
    "accuracy": accuracy_score(
        y_test,
        y_pred
    ),
    "precision": precision_score(
        y_test,
        y_pred,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        y_pred,
        zero_division=0
    ),
    "f1_score": f1_score(
        y_test,
        y_pred,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        y_prob
    )
}

df_metrics = pd.DataFrame(
    [
        {
            "metric": metric,
            "value": value,
            "execution_date": EXECUTION_DATE
        }
        for metric, value in (
            metrics.items()
        )
    ]
)

display(df_metrics)

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

## 11. Validação cruzada

A validação cruzada avalia a estabilidade do modelo.

In [0]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    classification_pipeline,
    X,
    y,
    cv=cv,
    scoring="f1"
)

df_cv_scores = pd.DataFrame({
    "fold": list(
        range(
            1,
            len(cv_scores) + 1
        )
    ),
    "f1_score": cv_scores
})

display(df_cv_scores)

print(
    "F1 médio:",
    round(
        cv_scores.mean(),
        4
    )
)

print(
    "Desvio padrão:",
    round(
        cv_scores.std(),
        4
    )
)

## 12. Matriz de confusão

In [0]:
cm = confusion_matrix(
    y_test,
    y_pred
)

df_confusion_matrix = pd.DataFrame(
    cm,
    index=[
        "real_sem_risco",
        "real_com_risco"
    ],
    columns=[
        "previsto_sem_risco",
        "previsto_com_risco"
    ]
)

display(
    df_confusion_matrix
)

fig, ax = plt.subplots(
    figsize=(6, 5)
)

image = ax.imshow(cm)

ax.set_title(
    "Matriz de Confusão"
)

ax.set_xlabel(
    "Classe prevista"
)

ax.set_ylabel(
    "Classe real"
)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.tight_layout()
plt.show()

## 13. Importância das variáveis

In [0]:
model = (
    classification_pipeline
    .named_steps["model"]
)

feature_importance = (
    model.feature_importances_
)

df_feature_importance = (
    pd.DataFrame({
        "feature": features,
        "importance": feature_importance
    })
    .sort_values(
        "importance",
        ascending=False
    )
    .reset_index(drop=True)
)

display(df_feature_importance)

fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.barh(
    df_feature_importance[
        "feature"
    ][::-1],
    df_feature_importance[
        "importance"
    ][::-1]
)

ax.set_title(
    "Importância das Variáveis"
)

ax.set_xlabel(
    "Importância"
)

plt.tight_layout()
plt.show()

## 14. Predição para toda a base

In [0]:
df_model_results = (
    df_ml[
        [
            "ano",
            "id_municipio",
            "sigla_uf"
        ]
    ]
    .copy()
)

df_model_results[
    "risco_real"
] = y.values

df_model_results[
    "risco_previsto"
] = (
    classification_pipeline
    .predict(X)
)

df_model_results[
    "probabilidade_risco"
] = (
    classification_pipeline
    .predict_proba(X)[:, 1]
)

df_model_results[
    "faixa_probabilidade"
] = pd.cut(
    df_model_results[
        "probabilidade_risco"
    ],
    bins=[
        -np.inf,
        0.25,
        0.50,
        0.75,
        np.inf
    ],
    labels=[
        "baixo",
        "moderado",
        "alto",
        "critico"
    ]
)

display(
    df_model_results
    .sort_values(
        "probabilidade_risco",
        ascending=False
    )
    .head(20)
)

## 15. Preparação da clusterização

A clusterização utiliza as mesmas features numéricas.

O objetivo é identificar grupos de municípios com padrões semelhantes.

In [0]:
cluster_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

X_cluster = (
    cluster_pipeline
    .fit_transform(
        df_ml[features]
    )
)

print(
    "Base preparada para clusterização:",
    X_cluster.shape
)

## 16. Seleção do número de clusters

In [0]:
silhouette_results = []

for k in range(2, 7):

    model_k = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    labels_k = (
        model_k.fit_predict(
            X_cluster
        )
    )

    score_k = silhouette_score(
        X_cluster,
        labels_k
    )

    silhouette_results.append({
        "k": k,
        "silhouette_score": score_k
    })

df_silhouette = pd.DataFrame(
    silhouette_results
)

display(df_silhouette)

best_k = int(
    df_silhouette
    .sort_values(
        "silhouette_score",
        ascending=False
    )
    .iloc[0]["k"]
)

print(
    "Quantidade selecionada de clusters:",
    best_k
)

## 17. Treinamento do K-Means

In [0]:
kmeans_model = KMeans(
    n_clusters=best_k,
    random_state=42,
    n_init=20
)

cluster_labels = (
    kmeans_model
    .fit_predict(
        X_cluster
    )
)

df_clusters = (
    df_ml[
        [
            "ano",
            "id_municipio",
            "sigla_uf"
        ]
        + features
    ]
    .copy()
)

df_clusters[
    "cluster"
] = cluster_labels

display(
    df_clusters.head()
)

## 18. Perfil dos clusters

Cada cluster é descrito pela média das features.

In [0]:
df_cluster_profile = (
    df_clusters
    .groupby("cluster")[
        features
    ]
    .mean()
    .reset_index()
)

df_cluster_size = (
    df_clusters
    .groupby("cluster")
    .size()
    .reset_index(
        name="quantidade_municipios"
    )
)

df_cluster_profile = (
    df_cluster_profile
    .merge(
        df_cluster_size,
        on="cluster",
        how="left"
    )
)

display(df_cluster_profile)

## 19. Classificação dos clusters por vulnerabilidade

A vulnerabilidade é determinada principalmente por:

- menor taxa de alfabetização;
- menor proficiência;
- maior gap para meta;
- menor participação.

In [0]:
def encontrar_coluna(
    candidatas
):
    for column in candidatas:
        if column in df_cluster_profile.columns:
            return column

    return None


col_taxa = encontrar_coluna([
    "PC_ALUNO_ALFABETIZADO",
    "taxa_alfabetizacao_alunos"
])

col_gap = encontrar_coluna([
    "gap_meta_2030"
])

col_proficiencia = encontrar_coluna([
    "VL_MEDIA_LP",
    "proficiencia_media_lp"
])

df_cluster_risk = (
    df_cluster_profile.copy()
)

df_cluster_risk[
    "score_vulnerabilidade"
] = 0.0

if col_taxa:
    df_cluster_risk[
        "score_vulnerabilidade"
    ] += (
        100
        - df_cluster_risk[col_taxa]
    )

if col_gap:
    df_cluster_risk[
        "score_vulnerabilidade"
    ] += (
        df_cluster_risk[col_gap]
        .clip(lower=0)
    )

if col_proficiencia:
    max_prof = (
        df_cluster_risk[
            col_proficiencia
        ]
        .max()
    )

    df_cluster_risk[
        "score_vulnerabilidade"
    ] += (
        max_prof
        - df_cluster_risk[
            col_proficiencia
        ]
    )

df_cluster_risk = (
    df_cluster_risk
    .sort_values(
        "score_vulnerabilidade",
        ascending=False
    )
    .reset_index(drop=True)
)

labels_vulnerabilidade = [
    "critico",
    "alto",
    "moderado",
    "baixo",
    "muito_baixo",
    "resiliente"
]

df_cluster_risk[
    "nivel_vulnerabilidade"
] = labels_vulnerabilidade[
    :len(df_cluster_risk)
]

display(df_cluster_risk)

## 20. Associação do perfil aos municípios

In [0]:
cluster_labels_df = (
    df_cluster_risk[
        [
            "cluster",
            "nivel_vulnerabilidade",
            "score_vulnerabilidade"
        ]
    ]
)

df_clusters = (
    df_clusters
    .merge(
        cluster_labels_df,
        on="cluster",
        how="left"
    )
)

display(
    df_clusters
    .sort_values(
        [
            "nivel_vulnerabilidade",
            "sigla_uf"
        ]
    )
    .head(20)
)

## 21. Matriz de risco por UF

A matriz consolida:

- quantidade de municípios;
- probabilidade média de risco;
- taxa média de risco real;
- distribuição por vulnerabilidade.

In [0]:
df_risk_matrix = (
    df_model_results
    .merge(
        df_clusters[
            [
                "ano",
                "id_municipio",
                "cluster",
                "nivel_vulnerabilidade"
            ]
        ],
        on=[
            "ano",
            "id_municipio"
        ],
        how="left"
    )
    .groupby(
        [
            "ano",
            "sigla_uf",
            "nivel_vulnerabilidade"
        ],
        dropna=False
    )
    .agg(
        quantidade_municipios=(
            "id_municipio",
            "nunique"
        ),
        probabilidade_media_risco=(
            "probabilidade_risco",
            "mean"
        ),
        taxa_risco_real=(
            "risco_real",
            "mean"
        ),
        taxa_risco_previsto=(
            "risco_previsto",
            "mean"
        )
    )
    .reset_index()
)

display(
    df_risk_matrix
    .sort_values(
        "probabilidade_media_risco",
        ascending=False
    )
)

## 22. Persistência dos resultados de IA

In [0]:
df_model_results.to_csv(
    IA_MODEL_RESULTS_PATH
    / "MODEL_RESULTS.csv",
    sep=";",
    decimal=",",
    encoding="utf-8",
    index=False
)

df_feature_importance.to_csv(
    IA_FEATURE_IMPORTANCE_PATH
    / "FEATURE_IMPORTANCE.csv",
    sep=";",
    decimal=",",
    encoding="utf-8",
    index=False
)

df_clusters.to_csv(
    IA_CLUSTERS_PATH
    / "MUNICIPIOS_CLUSTERS.csv",
    sep=";",
    decimal=",",
    encoding="utf-8",
    index=False
)

df_cluster_risk.to_csv(
    IA_CLUSTERS_PATH
    / "CLUSTER_PROFILES.csv",
    sep=";",
    decimal=",",
    encoding="utf-8",
    index=False
)

df_risk_matrix.to_csv(
    IA_RISK_MATRIX_PATH
    / "MATRIZ_RISCO_UF.csv",
    sep=";",
    decimal=",",
    encoding="utf-8",
    index=False
)

df_metrics.to_csv(
    IA_LOG_PATH
    / "MODEL_METRICS.csv",
    sep=";",
    decimal=",",
    encoding="utf-8",
    index=False
)

print(
    "Resultados de IA persistidos "
    "com sucesso."
)

## 23. Exportação consolidada para Power BI

In [0]:
df_ia_powerbi = (
    df_model_results
    .merge(
        df_clusters[
            [
                "ano",
                "id_municipio",
                "cluster",
                "nivel_vulnerabilidade",
                "score_vulnerabilidade"
            ]
        ],
        on=[
            "ano",
            "id_municipio"
        ],
        how="left"
    )
)

df_ia_powerbi.to_csv(
    IA_POWERBI_PATH
    / "POWERBI_IA_DASHBOARD.csv",
    sep=";",
    decimal=",",
    encoding="utf-8",
    index=False
)

print(
    "Base Power BI de IA salva em:",
    IA_POWERBI_PATH
)

## 24. Checklist final

A etapa será considerada concluída quando:

- o modelo possuir as duas classes;
- as métricas forem calculadas;
- a importância das variáveis estiver disponível;
- a clusterização possuir score de silhueta válido;
- os resultados forem persistidos.

In [0]:
if np.isnan(
    metrics["roc_auc"]
):
    raise ValueError(
        "ROC AUC não pôde ser calculado."
    )

if df_feature_importance.empty:
    raise ValueError(
        "Feature Importance vazia."
    )

if df_clusters["cluster"].nunique() < 2:
    raise ValueError(
        "Clusterização inválida."
    )

if df_risk_matrix.empty:
    raise ValueError(
        "Matriz de risco vazia."
    )

print(
    "IA e Modelagem concluídas "
    "com sucesso."
)

print(
    "F1 Score:",
    round(
        metrics["f1_score"],
        4
    )
)

print(
    "ROC AUC:",
    round(
        metrics["roc_auc"],
        4
    )
)

print(
    "Clusters:",
    best_k
)

## Resultado esperado

Ao final deste notebook estarão disponíveis:

```text
gold/ia/model_results/
gold/ia/feature_importance/
gold/ia/clusters/
gold/ia/risk_matrix/
gold/exports_powerbi/ia_dashboard/
logs/ia/model_metrics/
```

### Próxima etapa

```text
09_documentacao
```